# 01 – Exploratory data analysis
E-Commerce churn dataset (Kaggle). Goal: understand the data before building the pipeline in `churn/`.
Figures are saved to `reports/figures/eda_*.png`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from churn import config
from churn.dataset import find_raw_file, load_raw, clean

raw = load_raw(find_raw_file())
raw.shape

## Data quality: missing values, duplicates, inconsistent labels

In [ ]:
print(raw.isna().sum()[raw.isna().sum() > 0])
cols = [c for c in raw.columns if c != config.ID_COL]
print('rows identical except ID:', raw.duplicated(cols).sum())
for c in config.CATEGORICAL_FEATURES:
    print(c, raw[c].value_counts().to_dict())

In [ ]:
df = clean(raw)
print(df.shape, 'churn rate = %.1f%%' % (100 * df[config.TARGET].mean()))

## Target balance

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
df[config.TARGET].map({0: 'Stays', 1: 'Churns'}).value_counts().plot.bar(ax=ax, color=['#2E75B6', '#C0504D'])
ax.set_title('Churn distribution'); ax.set_xlabel(''); plt.xticks(rotation=0)
fig.savefig(config.FIGURES_DIR / 'eda_target.png', dpi=130, bbox_inches='tight')

## Churn rate by key variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), ['Complain', 'SatisfactionScore', 'CityTier', 'PreferedOrderCat', 'MaritalStatus', 'PreferredPaymentMode']):
    (df.groupby(col)[config.TARGET].mean() * 100).plot.bar(ax=ax, color='#2E75B6')
    ax.set_ylabel('churn rate (%)'); ax.set_title(col); ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'eda_churn_by_category.png', dpi=130, bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ['Tenure', 'CashbackAmount', 'DaySinceLastOrder']):
    sns.kdeplot(data=df, x=col, hue=config.TARGET, common_norm=False, fill=True, ax=ax)
    ax.set_title(col)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'eda_numeric_distributions.png', dpi=130, bbox_inches='tight')

In [ ]:
corr = df[config.NUMERIC_FEATURES + [config.TARGET]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax, annot=False)
ax.set_title('Correlation matrix')
fig.savefig(config.FIGURES_DIR / 'eda_correlation.png', dpi=130, bbox_inches='tight')
corr[config.TARGET].drop(config.TARGET).sort_values()

## Takeaways
* ~17% churners → imbalanced classes: use class weights and recall / PR-AUC, not accuracy.
* 7 numeric columns have ~5% missing values → median imputation inside the model pipeline.
* Inconsistent labels (Phone / Mobile Phone, CC / Credit Card, COD / Cash on Delivery) → harmonised in `clean()`.
* 557 rows (after harmonising the labels) are copies of other rows apart from the ID → removed to avoid train/test leakage.
* Strongest signals: short tenure, complaints, low cashback, single customers, few days since last order.